# Amazon Bedrock AgentCore Policy - NL2Cedar 데모

## 개요

Amazon Bedrock AgentCore Policy 실습 데모에 오신 것을 환영합니다! 이 Notebook에서는 자연어로 Cedar 정책을 생성하는 전체 워크플로를 살펴봅니다. 생성할 수 있는 다양한 정책 유형과 정책 구조를 이해하는 방법도 알아봅니다.

### Amazon Bedrock AgentCore Policy의 Natural Language Authoring이란?

NL2Cedar(Natural Language Authoring of Cedar Policies)를 사용하면 권한 부여 요구 사항을 자연어로 작성할 수 있습니다. 작성한 요구 사항은 Cedar 구문으로 자동 변환되며, 생성된 정책이 요구 사항과 일치하는지도 검증됩니다. 

---

## 사전 요구 사항

시작하기 전에 다음 사항을 확인하세요.

- ✅ 적절한 자격 증명으로 구성된 AWS CLI
- ✅ boto3가 설치된 Python 3.10 이상
- ✅ 설치된 `bedrock_agentcore_starter_toolkit` 패키지
- ✅ AWS Lambda 액세스 권한(대상 함수용)

01-Getting-Started/AgentCore-Policy-Demo.ipynb에서는 보험 인수 심사 사용 사례를 위한 Gateway와 Lambda 대상 3개를 설정합니다. 여기서도 동일한 Gateway 구성을 사용합니다. 

시작해 보겠습니다! 🚀

---

# Step 0: 환경 설정

먼저 환경을 확인하고 필요한 라이브러리를 가져옵니다.

In [ ]:
%pip install -r requirements.txt

In [ ]:
# 필수 라이브러리 가져오기
import sys
import os
from pathlib import Path
import subprocess
import boto3
import json
import logging
import time

# Python 경로에 scripts 디렉터리 추가
scripts_dir = Path.cwd() / "scripts"
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

# 리전 입력 요청
session = boto3.Session()
region = session.region_name
if not region:
    region = input("Enter AWS region (e.g., us-east-1, us-west-2): ").strip()
    if not region:
        raise ValueError("AWS region is required")

print(f"Region: {region}")

# AWS 자격 증명 확인
try:
    sts = session.client("sts", region_name=region)
    identity = sts.get_caller_identity()
    print("✅ AWS Credentials Verified")
    print(f"   Account: {identity['Account']}")
    print(f"   User/Role: {identity['Arn']}")
except Exception as e:
    print(f"❌ AWS Credentials Error: {e}")
    print("   Please configure AWS CLI with: aws configure")

### 08-AgentCore-policy/01-Getting-Started/AgentCore-Policy-Demo.ipynb의 Gateway 구성이 있는지 확인합니다. 구성이 없으면 다음 단계에서 Lambda 대상 3개를 포함하는 보험 인수 심사용 Gateway를 설정합니다.

In [ ]:
# 보험 인수 심사용 Gateway가 있는지 확인

# Getting-Started 디렉터리 경로 가져오기
current_dir = Path.cwd()
getting_started_dir = current_dir.parent / "01-Getting-Started"
config_file = getting_started_dir / "config.json"
scripts_dir = getting_started_dir / "scripts"

print("🔍 Checking for configuration...")
print(f"Looking for: {config_file.relative_to(current_dir.parent.parent)}")

if config_file.exists():
    print("✅ Configuration file found!")

    # 선택적으로 필수 필드가 있는지 확인
    import json

    try:
        with open(config_file, "r") as f:
            config = json.load(f)

        required_fields = ["lambdas", "gateway", "region"]
        missing_fields = [field for field in required_fields if field not in config]

        if missing_fields:
            print(f"⚠️  Warning: Config file is missing fields: {missing_fields}")
            print("   You may need to re-run the setup scripts.")
        else:
            print("✅ Configuration is complete!")

    except json.JSONDecodeError:
        print("⚠️  Warning: Config file exists but is not valid JSON")

else:
    print("❌ Configuration file not found!")
    print("\n" + "=" * 70)
    print("Setting up infrastructure...")
    print("=" * 70)

    # 스크립트를 실행하도록 Getting-Started 디렉터리로 이동
    os.chdir(getting_started_dir)

    try:
        # Step 1: Lambda 함수 배포
        print("\n📦 Step 1: Deploying Lambda functions...")
        print("-" * 70)
        deploy_lambda_script = scripts_dir / "lambda-target-setup" / "deploy_lambdas.py"
        result = subprocess.run([sys.executable, str(deploy_lambda_script)], capture_output=True, text=True)
        print(result.stdout)

        # Step 2: Gateway 설정
        print("\n🌐 Step 2: Setting up AgentCore Gateway...")
        print("-" * 70)
        setup_gateway_script = scripts_dir / "setup_gateway.py"
        result = subprocess.run([sys.executable, str(setup_gateway_script)], capture_output=True, text=True)
        print(result.stdout)

        print("\n" + "=" * 70)
        print("✅ Infrastructure setup complete!")
        print("=" * 70)

        # 구성 파일이 생성되었는지 확인
        if config_file.exists():
            print(f"✅ Configuration file created: {config_file}")
        else:
            print("⚠️  Warning: Setup completed but config.json was not created")

    except Exception as e:
        print(f"\n❌ Setup failed: {e}")
        print("\nPlease run the setup scripts manually:")
        print(f"1. cd {getting_started_dir}")
        print("2. python scripts/lambda-target-setup/deploy_lambdas.py")
        print("3. python scripts/setup_gateway.py")
    finally:
        # 원래 디렉터리로 돌아가기
        os.chdir(current_dir)

print("\n" + "=" * 70)

---

# Step 1: Policy Engine 생성

이제 Cedar 정책을 저장할 Policy Engine을 생성합니다.

Policy Engine은 정책의 모음입니다. Gateway와 연결하여 유입 트래픽에 정책을 실시간으로 적용할 수 있습니다.
다음 단계부터 이 Policy Engine에 정책을 생성합니다.

### Policy Engine 생성

먼저 Cedar 정책을 저장할 Policy Engine을 생성합니다.

In [ ]:
from bedrock_agentcore_starter_toolkit.operations.policy.client import PolicyClient

policy_client = PolicyClient(region_name=region)
policy_client.logger.setLevel(logging.INFO)

# Policy Engine 생성
print("🔧 Creating Policy Engine...")
engine = policy_client.create_or_get_policy_engine(
    name="InsurancePolicyEngine_NL2Cedar",
    description="Policy engine for insurance underwriting governance",
)
print(f"✓ Policy Engine: {engine['policyEngineId']}\n")

# 구성 파일에 Policy Engine 저장
with open(config_file, "r") as f:
    config = json.load(f)

# 기존 데이터를 제거하지 않고 Policy Engine 정보 추가
config["policy_engine_id"] = engine["policyEngineId"]
config["policy_engine_arn"] = engine["policyEngineArn"]

# 업데이트된 구성을 다시 기록
with open(config_file, "w") as f:
    json.dump(config, f, indent=2)

## Step 2: 자연어로 정책 생성

이제 NL2Cedar 기능을 사용하여 자연어로 Cedar 정책을 생성합니다. NL2Cedar를 사용한 정책 생성은 두 단계로 진행됩니다. 먼저 자연어로 Cedar 정책을 생성한 다음, 해당 정책을 Policy Engine에 생성합니다. 

> **💡 팁**: Foundation model이 대상 이름과 파라미터를 이해할 수 있도록 Gateway 대상의 스키마가 NL2Cedar 기능에 제공됩니다.

Gateway에는 다음과 같은 Lambda 대상 3개가 있습니다.
1. Application Tool: 간소화된 신청 생성(데모용 모의 구현)
 신청자의 리전과 보장 금액을 사용하여 보험 신청을 생성합니다.
 파라미터:
 - applicant_region: 고객의 지리적 리전
 - coverage_amount: 요청한 보험 보장 금액

2. Risk Model Tool: 간소화된 위험 모델 액세스(데모용 모의 구현)
 위험 점수 모델을 호출하고 평가 결과를 반환합니다.
 파라미터:
 - API_classification: API 분류(`public`, `internal`, `restricted`)
 - data_governance_approval: 데이터 거버넌스에서 모델 사용을 승인했는지 여부

3. Approval Tool: 보험 승인 절차(데모용 모의 구현)
 인수 결정과 청구 금액을 승인합니다.
 파라미터:
 - claim_amount: 보험 청구/보장 금액
 - risk_level: 위험 수준 평가(`low`, `medium`, `high`, `critical`)


 자연어 정책문에서 이러한 대상을 참조하고 파라미터를 기반으로 Agent의 도구 액세스에 제약 조건을 적용할 수 있습니다. 

### Cedar 정책 검증 결과

NL2Cedar는 생성한 Cedar 정책을 반환하기 전에 여러 단계의 검증 프로세스를 실행합니다. 이 과정에서 정책의 문제를 설명하는 **검증 결과(findings)**가 생성될 수 있습니다.

| 검증 결과 유형 | 의미 |
|---|---|
| `INVALID` | Cedar 구문 또는 스키마 오류 - 엄격 모드에서 정책 생성을 차단할 수 있음 |
| `WARNING` | 차단하지 않는 문제 - 정책의 구문은 유효하지만 예상과 다르게 동작할 수 있음 |

심각한 문제가 있는 자산을 서비스에서 반환하지 않으면 검증 결과로 인해 `generatedPolicies` 목록이 비어 있을 수 있습니다. 자산이 있더라도 검증 결과가 존재하면 `create_policy` 호출에서 검증 오류가 발생할 수 있습니다.

검증 결과와 관계없이 정책을 생성하려면 `create_policy` 또는 `create_or_get_policy`에 `validation_mode="IGNORE_ALL_FINDINGS"`를 전달합니다. 이 Notebook에서는 기본 생성이 실패한 후 시도하는 **대체 방법으로만** 이 설정을 사용합니다.

> 📖 각 검증 결과 유형과 해결 지침에 대한 자세한 내용은 [Policy 생성 검증 결과](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/policy-generation-validation.html#policy-generation-findings)를 참조하세요.

In [ ]:
# Cedar 정책 생성
print("\n\U0001f4dd Generating Cedar Policy from Natural language...")
print("\n\U0001f4dd Simple natural language statement")

nl_input = "Allow all users to invoke the application tool when the coverage amount is under 1000000 and the applicant region is US or CA"

result = policy_client.generate_policy(
    policy_engine_id=config["policy_engine_id"],
    name=f"nl_policy_{int(time.time())}",
    resource={"arn": config["gateway"]["gateway_arn"]},
    content={"rawText": nl_input},
    fetch_assets=True,
)

In [ ]:
cedar_statement = None
if result.get("status") == "GENERATED" and result.get("generatedPolicies"):
    generated_policy = result["generatedPolicies"][0]
    findings = generated_policy.get("findings", [])
    invalid = [f for f in findings if f.get("type") == "INVALID"]
    warnings = [f for f in findings if f.get("type") == "WARNING"]
    if invalid:
        print(
            "⚠️  Policy generation returned INVALID findings (policy creation will be retried with IGNORE_ALL_FINDINGS):"
        )
        for f in invalid:
            print(f"   • {f.get('description', 'Unknown')}")
    if warnings:
        print(
            "⚠️  Policy generation returned WARNING findings (policy creation may be retried with IGNORE_ALL_FINDINGS):"
        )
        for f in warnings:
            print(f"   • {f.get('description', 'Unknown')}")
    cedar_statement = generated_policy.get("definition", {}).get("cedar", {}).get("statement")
    if cedar_statement:
        print("Generated Cedar Policy:")
        print("=" * 60)
        print(cedar_statement)
        print("=" * 60)
    else:
        print("⚠️  No Cedar statement in generated policy")
        print("Raw asset:", generated_policy)
elif not result.get("generatedPolicies"):
    print(
        "⚠️  generatedPolicies list is empty — NL2Cedar produced no assets (possibly due to findings blocking generation)"
    )

## Step 2: 생성된 Cedar 정책으로 Policy 생성

In [ ]:
if cedar_statement:
    try:
        application_creation_policy = policy_client.create_or_get_policy(
            policy_engine_id=config["policy_engine_id"],
            name="application_creation_policy",
            description="Allow application creation when coverage is under $1M and region is US or CA",
            definition={"cedar": {"statement": cedar_statement}},
        )
        print(f"✓ Policy ready: {application_creation_policy['policyId']}")
    except Exception as e:
        # 생성 실패(예: 검증 결과로 인해 차단됨) - IGNORE_ALL_FINDINGS로 재시도
        print(f"⚠️  Policy creation failed: {e}")
        print("   Retrying with validation_mode='IGNORE_ALL_FINDINGS'...")
        application_creation_policy = policy_client.create_or_get_policy(
            policy_engine_id=config["policy_engine_id"],
            name="application_creation_policy",
            description="Allow application creation when coverage is under $1M and region is US or CA",
            definition={"cedar": {"statement": cedar_statement}},
            validation_mode="IGNORE_ALL_FINDINGS",
        )
        print(f"✓ Policy ready with IGNORE_ALL_FINDINGS: {application_creation_policy['policyId']}")
else:
    print("⚠️  Skipping policy creation: no Cedar statement was generated")

---

# 자연어를 사용한 다른 정책 생성 유형

### 1. 여러 줄 정책문
여러 줄로 된 정책문을 제공하면 여러 정책이 생성되어 `result['generatedPolicies']`에 포함됩니다. 일관되게 사용된 구분 기호(쉼표, 마침표, 세미콜론 등)를 감지하여 개별 정책문을 구분합니다.


In [ ]:
print("\n📝 Multi-line statement")

nl_input = """Allow all users to invoke the risk model tool when data governance approval is true. 
Block users from calling the application tool unless coverage amount is present"""

result = policy_client.generate_policy(
    policy_engine_id=config["policy_engine_id"],
    name=f"nl_policy_{int(time.time())}",
    resource={"arn": config["gateway"]["gateway_arn"]},
    content={"rawText": nl_input},
    fetch_assets=True,
)

if result.get("status") == "GENERATED" and result.get("generatedPolicies"):
    for generated_policy in result["generatedPolicies"]:
        cedar_statement = generated_policy.get("definition", {}).get("cedar", {}).get("statement", "N/A")

        print("Generated Cedar Policy:")
        print("=" * 60)
        print(cedar_statement)
        print("=" * 60)

### 2. Principal 관련 정책문
OAuth 액세스 토큰의 IdP 표현을 통해 전달되는 principal 범위를 기반으로 조건을 검증하는 정책을 생성할 수 있습니다. OAuth 토큰에 저장할 속성을 사용자 지정할 수 있으므로, NL2Cedar 생성 시 정확한 태그를 제공하면 올바른 Cedar 정책을 생성하는 데 도움이 됩니다.

In [ ]:
print("\n📝 Principal Scope statements")

nl_inputs = [
    'Allow principals with username "test-user" to invoke the risk model tool',
    str(
        'Forbid principals to access the approval tool unless they have the scope group:Controller <idp_claims>["scope"]</idp_claims>'
    ),
    str(
        'Block principals from using risk model tool and approval tool unless the principal has role "senior-adjuster"'
    ),
]

for nl_input in nl_inputs:
    result = policy_client.generate_policy(
        policy_engine_id=config["policy_engine_id"],
        name=f"nl_policy_{int(time.time())}",
        resource={"arn": config["gateway"]["gateway_arn"]},
        content={"rawText": nl_input},
        fetch_assets=True,
    )

    if result.get("status") == "GENERATED" and result.get("generatedPolicies"):
        for generated_policy in result["generatedPolicies"]:
            cedar_statement = generated_policy.get("definition", {}).get("cedar", {}).get("statement", "N/A")
            print("=" * 60)
            print(f"Natural Language: {nl_input}")
            print("Generated Cedar Policy:")
            print("=" * 60)
            print(cedar_statement)
            print("=" * 60)

# 리소스 정리

In [ ]:
from bedrock_agentcore_starter_toolkit.operations.gateway.client import GatewayClient
from bedrock_agentcore_starter_toolkit.operations.policy.client import PolicyClient

with open(config_file, "r") as f:
    config = json.load(f)

# 먼저 Policy Engine 정리
print("🧹 Cleaning up Policy Engine...")
policy_client = PolicyClient(region_name=config["region"])
policy_client.cleanup_policy_engine(config["policy_engine_id"])
print("✓ Policy Engine cleaned up\n")

# 그런 다음 Gateway 정리
print("🧹 Cleaning up Gateway...")
gateway_client = GatewayClient(region_name=config["region"])
gateway_client.cleanup_gateway(config["gateway"]["gateway_id"], config["gateway"]["client_info"])
print("✅ Cleanup complete!")